# TEMA 1-IA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, mean_squared_error, confusion_matrix
from scipy.stats import chi2_contingency
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

#### diagrame roz

In [ ]:
sns.set_theme(style="whitegrid", palette=["#ff00aa"])

# Task 1 - Exploratory Data Analysis (EDA)

## Citirea datelor

### Verifica ce tip are fiecare coloana si daca exista valori lipsa.

In [ ]:
df_train = pd.read_csv("CC_education_economy_train.csv")
df_test = pd.read_csv("CC_education_economy_test.csv")

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

display(df_train.head())

## Tipuri de coloane si valori lipsa

### Verifica ce tip are fiecare coloana si daca exista valori lipsa.

In [ ]:
print("=== Informatii generale ===")

info_df = pd.DataFrame({
    "Coloana": df_train.columns,
    "Non-Null": df_train.notna().sum().values,
    "Valori lipsa": df_train.isna().sum().values,
    "Procent lipsa (%)": (df_train.isna().sum() / len(df_train) * 100).values,
    "Tip": df_train.dtypes.values
})

display(info_df.sort_values("Valori lipsa", ascending=False))

numeric_cols = df_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = df_train.select_dtypes(include=["object", "string"]).columns.tolist()

types_summary = pd.DataFrame({
    "Tip": ["Numerice", "Categorice"],
    "Numar coloane": [len(numeric_cols), len(categorical_cols)],
    "Coloane": [
        ", ".join(numeric_cols),
        ", ".join(categorical_cols)
    ]
})

print("\n=== Rezumat tipuri coloane ===")
display(types_summary)

print("\n=== Valori lipsa (highlight) ===")
display(info_df.style.background_gradient(cmap="Reds"))

## Separarea coloanelor 

### Imparte coloanele in numerice si categoriale

In [ ]:
numeric_cols = df_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = df_train.select_dtypes(include=["object", "string"]).columns.tolist()

columns_summary = pd.DataFrame({
    "Tip coloana": ["Numerica"] * len(numeric_cols) + ["Categorica"] * len(categorical_cols),
    "Nume coloana": numeric_cols + categorical_cols
})

display(columns_summary)

## Statistici pentru atribute numerice

### Calculeaza medie, deviatie standard, min, max, percentila 25%, 50%, 75%

In [ ]:
numeric_stats = df_train[numeric_cols].describe().T
display(numeric_stats)

## Boxplot pentru atribute numerice

### Arata vizual plaja de valori Si posibili outlier

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import math

df_train = pd.read_csv("CC_education_economy_train.csv")

num_cols = df_train.select_dtypes(include=["number"]).columns
n = len(num_cols)

cols = 3  
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(12, 4 * rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].boxplot(df_train[col].dropna())
    axes[i].set_title(col)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

## Statistici pentru atribute categoriale

### Pentru fiecare coloana categoriala arata cate valori nenule și cate valori unice are

In [ ]:
cat_stats = pd.DataFrame({
    "non_null_count": df_train[categorical_cols].notna().sum(),
    "unique_values": df_train[categorical_cols].nunique()
})

display(cat_stats)

## Distributii pentru atribute categoriale

### Face cate un grafic bar pentru fiecare coloană categoriala

In [ ]:
for col in categorical_cols:
    plt.figure(figsize=(10, 4))
    df_train[col].value_counts(dropna=False).plot(kind="bar")
    plt.title(f"Distributia valorilor pentru {col}")
    plt.xlabel(col)
    plt.ylabel("Numar exemple")
    plt.xticks(rotation=45)
    plt.show()

## Echilibrul claselor pentru vacation

### Verifica daca targetul de clasificare este echilibrat sau nu

In [ ]:
vacation_counts = df_train["vacation"].value_counts()
vacation_percent = (df_train["vacation"].value_counts(normalize=True) * 100).round(2)

vacation_df = pd.DataFrame({
    "Clasa": vacation_counts.index,
    "Numar exemple": vacation_counts.values,
    "Procent (%)": vacation_percent.values
})

display(vacation_df)

plt.figure(figsize=(8, 5))

bars = plt.bar(
    vacation_df["Clasa"],
    vacation_df["Numar exemple"],
    edgecolor="black"
)

for bar, percent in zip(bars, vacation_df["Procent (%)"]):
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 300,
        f"{int(height)}\n({percent}%)",
        ha="center",
        va="bottom",
        fontsize=10
    )

plt.title("Distributia claselor pentru vacation", fontsize=14)
plt.xlabel("Vacation")
plt.ylabel("Numar exemple")
plt.xticks(rotation=30)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Corelatie intre atribute numerice

### Calculeaza corelatiile Pearson intre coloanele numerice

In [ ]:
corr_numeric = df_train[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_numeric, annot=True, fmt=".2f")
plt.title("Matrice de corelatie pentru atribute numerice")
plt.show()

display(corr_numeric)

## Atribut numeric vs clasificare

### Verifica relatia dintre un atribut numeric si targetul vacation

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_train, x="vacation", y="experience_years")
plt.title("experience_years in functie de vacation")
plt.xlabel("Vacation")
plt.ylabel("Experience years")
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_train, x="vacation", y="salary")
plt.title("salary in functie de vacation")
plt.xlabel("Vacation")
plt.ylabel("Salary")
plt.xticks(rotation=45)
plt.show()

## Atribut categorial vs clasificare

### Compara education_level cu vacation

In [ ]:
ct = pd.crosstab(df_train["education_level"], df_train["vacation"])

plt.figure(figsize=(10, 5))
sns.heatmap(ct, annot=True, fmt="d")
plt.title("education_level vs vacation")
plt.xlabel("Vacation")
plt.ylabel("Education level")
plt.show()

display(ct)

### Test Chi-Square

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

cat_cols_with_target = [
    'job_title', 'education_level', 'industry', 'company_size',
    'remote_work', 'location', 'skill_bracket', 'vacation'
]

p_matrix = pd.DataFrame(
    np.ones((len(cat_cols_with_target), len(cat_cols_with_target))),
    index=cat_cols_with_target,
    columns=cat_cols_with_target
)

for col1 in cat_cols_with_target:
    for col2 in cat_cols_with_target:
        if col1 != col2:
            ct = pd.crosstab(df_train[col1], df_train[col2])
            _, p, _, _ = chi2_contingency(ct)
            p_matrix.loc[col1, col2] = p


plt.figure(figsize=(10, 8))

sns.heatmap(
    p_matrix,
    annot=True,
    fmt=".3f",
    cmap="viridis_r",  
    vmin=0,
    vmax=1,
    square=True
)

plt.title("Chi-Square p-values între variabile categoriale\n(albastru = relație semnificativă)")
plt.tight_layout()
plt.show()

## Atribut numeric vs regresie

### Verifica ce coloane numerice sunt corelate cu salary

In [ ]:
salary_corr = df_train[numeric_cols].corr()["salary"].sort_values(ascending=False)

salary_corr_df = pd.DataFrame({
    "Atribut": salary_corr.index,
    "Coeficient corelatie": salary_corr.values.round(4)
})

display(salary_corr_df)

plt.figure(figsize=(8, 5))

data_plot = salary_corr_df[salary_corr_df["Atribut"] != "salary"]

bars = plt.bar(
    data_plot["Atribut"],
    data_plot["Coeficient corelatie"],
    edgecolor="black"
)

for bar, val in zip(bars, data_plot["Coeficient corelatie"]):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        f"{val:.2f}",
        ha="center",
        va="bottom",
        fontsize=10
    )

plt.title("Corelatia atributelor numerice cu salary", fontsize=14)
plt.ylabel("Coeficient Pearson")
plt.xticks(rotation=30)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Atribut categorial vs regresie

### Verifica salariul mediu în functie de education_level

In [ ]:
salary_by_education = df_train.groupby("education_level")["salary"].mean().sort_values()

plt.figure(figsize=(10, 5))
salary_by_education.plot(kind="bar")
plt.title("Media salary in functie de education_level")
plt.xlabel("Education level")
plt.ylabel("Salary mediu")
plt.xticks(rotation=45)
plt.show()

display(
    df_train.groupby("education_level")["salary"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

### Si în funcție de job_title

In [ ]:
salary_by_job = df_train.groupby("job_title")["salary"].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 5))
salary_by_job.plot(kind="bar")
plt.title("Media salary in functie de job_title")
plt.xlabel("Job title")
plt.ylabel("Salary mediu")
plt.xticks(rotation=45)
plt.show()

display(
    df_train.groupby("job_title")["salary"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

## Corelatie intre atribute categoriale

### Verifica relatia dintre doua coloane categoriale

In [ ]:
ct2 = pd.crosstab(df_train["job_title"], df_train["skill_bracket"])

plt.figure(figsize=(10, 5))
sns.heatmap(ct2, annot=True, fmt="d")
plt.title("job_title vs skill_bracket")
plt.xlabel("Skill bracket")
plt.ylabel("Job title")
plt.show()

display(ct2)

In [ ]:
chi2, p, dof, expected = chi2_contingency(ct2)

results_df = pd.DataFrame({
    "Chi2": [round(chi2, 2)],
    "df": [dof],
    "p-value": [round(p, 4)]
})

print("=== Rezultatul testului Chi-Square ===")
display(results_df)
print("\n=== Interpretare ===")

if p < 0.05:
    print(f"p-value = {p:.4f} < 0.05")
    print("Exista relatie intre job_title si skill_bracket")
else:
    print(f"p-value = {p:.4f} >= 0.05")
    print("NU exista relatie clara intre job_title si skill_bracket")

# Task 2 - Preprocesarea datelor

## Copie dataset

### Se creeaza copii ale dataset-urilor pentru a pastra datele originale nemodificate.

In [ ]:
df_train_preprocessed = df_train.copy()
df_test_preprocessed = df_test.copy()

## Valori lipsa

### Se verifica numarul si procentul valorilor lipsa pentru fiecare coloana

In [ ]:
missing_before = pd.DataFrame({
    "Coloana": df_train_preprocessed.columns,
    "Valori lipsa": df_train_preprocessed.isna().sum().values,
    "Procent (%)": (df_train_preprocessed.isna().sum() / len(df_train_preprocessed) * 100).round(2).values
})

display(missing_before.sort_values("Valori lipsa", ascending=False))

## Imputare remote_work

### Valorile lipsa din `remote_work` sunt inlocuite cu valoarea cea mai frecventa, deoarece atributul este categorial

In [ ]:
from sklearn.impute import SimpleImputer

imputer_remote = SimpleImputer(strategy="most_frequent")

df_train_preprocessed["remote_work"] = imputer_remote.fit_transform(
    df_train_preprocessed[["remote_work"]]
).ravel()

df_test_preprocessed["remote_work"] = imputer_remote.transform(
    df_test_preprocessed[["remote_work"]]
).ravel()

missing_after = pd.DataFrame({
    "Coloana": df_train_preprocessed.columns,
    "Valori lipsa": df_train_preprocessed.isna().sum().values,
    "Procent (%)": (df_train_preprocessed.isna().sum() / len(df_train_preprocessed) * 100).round(2).values
})

display(missing_after.sort_values("Valori lipsa", ascending=False))

## Outlieri pe salary

### Outlierii din coloana `salary` sunt identificati folosind metoda IQR

In [ ]:
col = "salary"

Q1 = df_train_preprocessed[col].quantile(0.25)
Q3 = df_train_preprocessed[col].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_mask = (df_train_preprocessed[col] < lower_bound) | (df_train_preprocessed[col] > upper_bound)
nr_outliers = outliers_mask.sum()

outliers_df = pd.DataFrame({
    "Coloana": [col],
    "Q1": [Q1],
    "Q3": [Q3],
    "IQR": [IQR],
    "Limita inferioara": [lower_bound],
    "Limita superioara": [upper_bound],
    "Nr outlieri": [nr_outliers],
    "Procent outlieri (%)": [round(nr_outliers / len(df_train_preprocessed) * 100, 2)]
})

display(outliers_df)

## Grafic inainte de tratare

### Boxplot-ul este folosit pentru a observa valorile extreme din `salary` inainte de preprocesare

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(x=df_train_preprocessed["salary"], color="#ff00aa")
plt.title("Salary inainte de tratarea outlierilor")
plt.xlabel("Salary")
plt.show()

In [ ]:
median_salary = df_train_preprocessed["salary"].median()

df_train_preprocessed.loc[outliers_mask, "salary"] = median_salary

print(f"Outlierii din salary au fost inlocuiti cu mediana: {median_salary}")

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(x=df_train_preprocessed["salary"], color="#ff00aa")
plt.title("Salary dupa tratarea outlierilor")
plt.xlabel("Salary")
plt.show()

## Redundanta

### Se analizeaza corelatiile dintre atributele numerice pentru a verifica daca exista atribute redundante

In [ ]:
corr_after = df_train_preprocessed.select_dtypes(include=["number"]).corr()

plt.figure(figsize=(9, 6))
sns.heatmap(corr_after, annot=True, fmt=".2f", cmap="RdPu")
plt.title("Corelatie dupa tratarea datelor")
plt.show()

## Encoding

### Variabilele categoriale sunt transformate in coloane numerice folosind One-Hot Encoding

In [ ]:
target_classification = "vacation"
target_regression = "salary"

X_train = df_train_preprocessed.drop(columns=[target_classification, target_regression])
y_train_classification = df_train_preprocessed[target_classification]
y_train_regression = df_train_preprocessed[target_regression]

X_test = df_test_preprocessed.copy()

X_train_encoded = pd.get_dummies(X_train, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)

X_train_encoded, X_test_encoded = X_train_encoded.align(
    X_test_encoded,
    join="left",
    axis=1,
    fill_value=0
)

print("Dimensiune X_train dupa encoding:", X_train_encoded.shape)
print("Dimensiune X_test dupa encoding:", X_test_encoded.shape)

## Standardizare

### Atributele numerice sunt standardizate pentru a avea aceeasi scala

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()

X_train_encoded[numeric_features] = scaler.fit_transform(X_train_encoded[numeric_features])
X_test_encoded[numeric_features] = scaler.transform(X_test_encoded[numeric_features])

display(X_train_encoded.head())

#  Task 3 - Utilizarea algoritmilor de Invatare Automata

## Pregatirea datelor pentru modele

### Datele sunt impartite in set de antrenare si validare, iar targeturile sunt separate pentru clasificare si regresie.

In [ ]:
X_encoded = X_train_encoded.copy()
y_class = y_train_classification.copy()
y_reg = y_train_regression.copy()

label_encoder = LabelEncoder()
y_class_encoded = label_encoder.fit_transform(y_class)

X_train_model, X_val_model, y_train_class, y_val_class, y_train_reg, y_val_reg = train_test_split(
    X_encoded,
    y_class_encoded,
    y_reg,
    test_size=0.2,
    random_state=42,
    stratify=y_class_encoded
)
split_df = pd.DataFrame({
    "Set": ["Train", "Validation"],
    "Nr exemple": [len(X_train_model), len(X_val_model)],
    "Nr features": [X_train_model.shape[1], X_val_model.shape[1]]
})

display(split_df)


## Clasificare - Decision Tree baseline

### Se foloseste un arbore de decizie ca model baseline pentru clasificarea variabilei vacation

In [ ]:
dt_baseline = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

dt_baseline.fit(X_train_model, y_train_class)
y_pred_dt = dt_baseline.predict(X_val_model)

dt_results = pd.DataFrame({
    "Model": ["Decision Tree"],
    "max_depth": [5],
    "Accuracy": [round(accuracy_score(y_val_class, y_pred_dt), 4)],
    "Precision": [round(precision_score(y_val_class, y_pred_dt, average="macro"), 4)],
    "Recall": [round(recall_score(y_val_class, y_pred_dt, average="macro"), 4)],
    "F1": [round(f1_score(y_val_class, y_pred_dt, average="macro"), 4)]
})

display(dt_results)

report_dt = pd.DataFrame(
    classification_report(
        y_val_class,
        y_pred_dt,
        target_names=label_encoder.classes_,
        output_dict=True
    )
).transpose()

display(report_dt.round(3))

## Experimente clasificare - variere hiperparametru

### Se variaza hiperparametrul max_depth pentru a observa influenta adancimii arborelui asupra performantei

In [ ]:
classification_results = []

depth_values = [3, 5, 8, 10]

for depth in depth_values:
    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )
    
    model.fit(X_train_model, y_train_class)
    y_pred = model.predict(X_val_model)
    
    classification_results.append({
        "Model": "Decision Tree",
        "max_depth": depth,
        "Accuracy": round(accuracy_score(y_val_class, y_pred), 4),
        "Precision": round(precision_score(y_val_class, y_pred, average="macro"), 4),
        "Recall": round(recall_score(y_val_class, y_pred, average="macro"), 4),
        "F1": round(f1_score(y_val_class, y_pred, average="macro"), 4)
    })

classification_results_df = pd.DataFrame(classification_results)
display(classification_results_df)

## Model suplimentar clasificare - Logistic Regression

### Se foloseste regresia logistica pentru comparatie cu arborele de decizie

In [ ]:
log_reg = LogisticRegression(
    max_iter=5000,
    solver="saga",
    random_state=42
)

log_reg.fit(X_train_model, y_train_class)
y_pred_lr = log_reg.predict(X_val_model)

log_reg_results = {
    "Model": "Logistic Regression",
    "max_depth": "-",
    "Accuracy": round(accuracy_score(y_val_class, y_pred_lr), 4),
    "Precision": round(precision_score(y_val_class, y_pred_lr, average="macro"), 4),
    "Recall": round(recall_score(y_val_class, y_pred_lr, average="macro"), 4),
    "F1": round(f1_score(y_val_class, y_pred_lr, average="macro"), 4)
}

classification_results_df = pd.concat(
    [classification_results_df, pd.DataFrame([log_reg_results])],
    ignore_index=True
)

display(classification_results_df)

## Matrice de confuzie pentru cel mai bun model

### Matricea de confuzie arata clasele unde modelul face cele mai multe greseli

In [ ]:
best_idx = classification_results_df["F1"].idxmax()
best_model_info = classification_results_df.loc[best_idx]

best_model_df = pd.DataFrame({
    "Model ales": [best_model_info["Model"]],
    "Hiperparametru": ["max_depth" if best_model_info["Model"] == "Decision Tree" else "max_iter"],
    "Valoare": [best_model_info["max_depth"] if best_model_info["Model"] == "Decision Tree" else "5000"],
    "Accuracy": [round(best_model_info["Accuracy"], 4)],
    "Precision": [round(best_model_info["Precision"], 4)],
    "Recall": [round(best_model_info["Recall"], 4)],
    "F1": [round(best_model_info["F1"], 4)]
})

display(best_model_df)

if best_model_info["Model"] == "Decision Tree":
    best_classifier = DecisionTreeClassifier(
        max_depth=int(best_model_info["max_depth"]),
        random_state=42
    )
else:
    best_classifier = LogisticRegression(
        max_iter=5000,
        solver="saga",
        random_state=42
    )

best_classifier.fit(X_train_model, y_train_class)
y_pred_best = best_classifier.predict(X_val_model)

cm = confusion_matrix(y_val_class, y_pred_best)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="RdPu",
    linewidths=0.5,
    linecolor="white",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)

plt.title("Matrice de confuzie - cel mai bun model", fontsize=14, fontweight="bold")
plt.xlabel("Predictie", fontsize=12)
plt.ylabel("Valoare reala", fontsize=12)
plt.xticks(rotation=30)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Regresie - Linear Regression baseline

### Se foloseste regresia liniara ca model baseline pentru prezicerea variabilei salary

In [ ]:
linear_model = LinearRegression()

linear_model.fit(X_train_model, y_train_reg)

y_pred_train_lr = linear_model.predict(X_train_model)
y_pred_val_lr = linear_model.predict(X_val_model)

linear_results = {
    "Model": "Linear Regression",
    "Alpha": "-",
    "Train MAE": round(mean_absolute_error(y_train_reg, y_pred_train_lr), 2),
    "Val MAE": round(mean_absolute_error(y_val_reg, y_pred_val_lr), 2),
    "Train MSE": round(mean_squared_error(y_train_reg, y_pred_train_lr), 2),
    "Val MSE": round(mean_squared_error(y_val_reg, y_pred_val_lr), 2),
    "Train R2": round(r2_score(y_train_reg, y_pred_train_lr), 4),
    "Val R2": round(r2_score(y_val_reg, y_pred_val_lr), 4)
}

display(pd.DataFrame([linear_results]))

## Regresie - Ridge cu regularizare

### Se variaza factorul de regularizare alpha pentru a observa efectul asupra performantei modelului

In [ ]:
regression_results = [linear_results]

alpha_values = [0.1, 1, 10, 100]

for alpha in alpha_values:
    ridge = Ridge(alpha=alpha)
    
    ridge.fit(X_train_model, y_train_reg)
    
    y_pred_train = ridge.predict(X_train_model)
    y_pred_val = ridge.predict(X_val_model)
    
    regression_results.append({
        "Model": "Ridge Regression",
        "Alpha": alpha,
        "Train MAE": round(mean_absolute_error(y_train_reg, y_pred_train), 2),
        "Val MAE": round(mean_absolute_error(y_val_reg, y_pred_val), 2),
        "Train MSE": round(mean_squared_error(y_train_reg, y_pred_train), 2),
        "Val MSE": round(mean_squared_error(y_val_reg, y_pred_val), 2),
        "Train R2": round(r2_score(y_train_reg, y_pred_train), 4),
        "Val R2": round(r2_score(y_val_reg, y_pred_val), 4)
    })

regression_results_df = pd.DataFrame(regression_results)
display(regression_results_df)

## Grafic eroare train vs validation

### Graficul compara eroarea pe train si validation pentru modelele de regresie

In [ ]:
plot_df = regression_results_df.copy()
plot_df["Experiment"] = plot_df["Model"].astype(str) + " alpha=" + plot_df["Alpha"].astype(str)

plt.figure(figsize=(10, 5))

plt.plot(plot_df["Experiment"], plot_df["Train MSE"], marker="o", label="Train MSE")
plt.plot(plot_df["Experiment"], plot_df["Val MSE"], marker="o", label="Validation MSE")

plt.title("Comparatie eroare train vs validation")
plt.xlabel("Experiment")
plt.ylabel("MSE")
plt.xticks(rotation=45)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Cel mai bun model de regresie

### Cel mai bun model este ales pe baza erorii de validare Val MSE

In [ ]:
best_reg_idx = regression_results_df["Val MSE"].idxmin()
best_regression_model = regression_results_df.loc[best_reg_idx]

best_reg_df = pd.DataFrame({
    "Model ales": [best_regression_model["Model"]],
    "Alpha": [best_regression_model["Alpha"]],
    "Train MSE": [best_regression_model["Train MSE"]],
    "Val MSE": [best_regression_model["Val MSE"]],
    "Train R2": [best_regression_model["Train R2"]],
    "Val R2": [best_regression_model["Val R2"]]
})

display(best_reg_df)

## Evaluarea finala a algoritmilor

Aceasta sectiune compara modelele folosite pentru clasificare si regresie.  
Pentru clasificare, cel mai bun model este ales dupa F1-score.  
Pentru regresie, cel mai bun model este ales dupa eroarea minima pe validare.

In [ ]:
hyperparams_df = pd.DataFrame({
    "Algoritm": [
        "Decision Tree Classifier",
        "Logistic Regression",
        "Linear Regression",
        "Ridge Regression"
    ],
    "Hiperparametri finali": [
        "max_depth ales din experimente, random_state=42",
        "max_iter=5000, solver=saga, random_state=42",
        "default sklearn",
        "alpha ales din experimente"
    ]
})

display(hyperparams_df)
display(classification_results_df)
display(regression_results_df)



plt.figure(figsize=(8, 5))

plt.bar(
    classification_results_df["Model"].astype(str) + " " + classification_results_df["max_depth"].astype(str),
    classification_results_df["F1"],
    color="#ff00aa",
    edgecolor="black"
)

plt.title("Comparatie modele clasificare dupa F1-score")
plt.xlabel("Model")
plt.ylabel("F1-score")
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

plot_reg = regression_results_df.copy()
plot_reg["Experiment"] = plot_reg["Model"].astype(str) + " alpha=" + plot_reg["Alpha"].astype(str)

plt.figure(figsize=(9, 5))

plt.bar(
    plot_reg["Experiment"],
    plot_reg["Val MSE"],
    color="#ff00aa",
    edgecolor="black"
)

plt.title("Comparatie modele regresie dupa Validation MSE")
plt.xlabel("Model")
plt.ylabel("Validation MSE")
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## Kaggle

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression

df_train = pd.read_csv("CC_education_economy_train.csv")
df_private = pd.read_csv("CC_private_test.csv")

ids = df_private["id"].copy() if "id" in df_private.columns else pd.RangeIndex(start=0, stop=len(df_private))

y_cls = df_train["vacation"]
y_reg = df_train["salary"]

X_train_raw = df_train.drop(columns=["vacation", "salary", "id"], errors="ignore")
X_private_raw = df_private.drop(columns=["vacation", "salary", "id"], errors="ignore")

combined = pd.concat([X_train_raw, X_private_raw], axis=0)

if "remote_work" in combined.columns:
    combined["remote_work"] = combined["remote_work"].fillna(combined["remote_work"].mode()[0])

combined_encoded = pd.get_dummies(combined, drop_first=True)

X_train_encoded = combined_encoded.iloc[:len(X_train_raw)].copy()
X_private_encoded = combined_encoded.iloc[len(X_train_raw):].copy()

num_cols = [
    "experience_years",
    "skills_count",
    "certifications",
    "total_days_worked",
    "aggregated_score"
]

num_cols = [col for col in num_cols if col in X_train_encoded.columns]

scaler = StandardScaler()
X_train_encoded[num_cols] = scaler.fit_transform(X_train_encoded[num_cols])
X_private_encoded[num_cols] = scaler.transform(X_private_encoded[num_cols])

log_reg_final = LogisticRegression(
    max_iter=5000,
    solver="saga",
    random_state=42
)

log_reg_final.fit(X_train_encoded, y_cls)
y_pred_cls = log_reg_final.predict(X_private_encoded)

submission_cls = pd.DataFrame({
    "id": ids,
    "vacation": y_pred_cls
})

submission_cls.to_csv("submission_clasificare.csv", index=False)

lin_reg_final = LinearRegression()

lin_reg_final.fit(X_train_encoded, y_reg)
y_pred_reg = lin_reg_final.predict(X_private_encoded)

submission_reg = pd.DataFrame({
    "id": ids,
    "salary": y_pred_reg
})

submission_reg.to_csv("submission_regresie.csv", index=False)

print("Gata:")
print("submission_clasificare.csv")
print("submission_regresie.csv")

display(submission_cls.head())
display(submission_reg.head())